In [2]:
import numpy as np
import pandas as pd
import json
from itertools import combinations
import itertools
import random
from collections import defaultdict
import os
import shutil
import ast
from matplotlib import pyplot as plt
import pickle
import json
import random
from collections import defaultdict
from itertools import combinations
from itertools import permutations
from itertools import product
import pickle
from scipy.stats import kendalltau
from scipy.stats import spearmanr
from scipy.stats import pearsonr
from scipy.stats import mannwhitneyu
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import mutual_info_score
from statsmodels.stats.multitest import multipletests
import seaborn as sns
import matplotlib.pyplot as plt
import re

### Preparation work

In [3]:
# Read in the dataframes
def DownloadAndRenaming(path):
    df = pd.read_csv(path)

    if path == "/Users/guowanyi/Desktop/spring 2026/Louis_Vuitton/Jan 28 update/Gene_Mutation_Trimmed.csv":
        df = df.rename(columns={"gene name": "Gene"})
        df = df.set_index("Gene")

    else:
        df = df.rename(columns={"Unnamed: 0": "Gene"})
        df = df.set_index("Gene")
    
    return df

cancer_complexes = pd.read_excel('/Users/guowanyi/Desktop/fall_2025/Louis_Vuitton/Nov24_update_required_files/cancer_complexes_clean.xlsx', index_col = 0)
cancer_complexes["Representative Genes (Core Members)"] = (cancer_complexes["Representative Genes (Core Members)"].apply(ast.literal_eval))


Gene_expression = DownloadAndRenaming('/Users/guowanyi/Desktop/fall_2025/Louis_Vuitton/Nov24_update_required_files/DepMap_Trimmed/Gene_Expression_Trimmed.csv')
shRNA = DownloadAndRenaming('/Users/guowanyi/Desktop/fall_2025/Louis_Vuitton/Nov24_update_required_files/DepMap_Trimmed/shRNA_Trimmed.csv')
CRISPR = DownloadAndRenaming('/Users/guowanyi/Desktop/fall_2025/Louis_Vuitton/Nov24_update_required_files/DepMap_Trimmed/CRISPR_Trimmed.csv')
Gene_mutation = DownloadAndRenaming('/Users/guowanyi/Desktop/spring 2026/Louis_Vuitton/Jan 28 update/Gene_Mutation_Trimmed.csv')
copy_number = DownloadAndRenaming('/Users/guowanyi/Desktop/spring 2026/Louis_Vuitton/Jan 28 update/Copy_Number_Trimmed.csv')

In [8]:
# Read in the positive controls
with open('/Users/guowanyi/Desktop/fall_2025/Louis_Vuitton/Nov24_update_required_files/Ten_positive_controls_1119.pkl', "rb") as f:
    Ten_positive_controls_1119 = pickle.load(f)

In [7]:
# Read in the fixed negative controls
with open('/Users/guowanyi/Desktop/spring 2026/Louis_Vuitton/Feb 25 update/Fixed_10_negative_controls0223.pkl', "rb") as f:
    Fixed_10_negative_controls = pickle.load(f)

In [9]:
# Convert the control dictionary to data frame
def convert2genePairDF(control_dict, controlType = ["positive", "negative"]):

    if isinstance(control_dict, pd.DataFrame):
        genePairDF = control_dict.copy()

    else: 

        if controlType == "negative":
            genePairDF = pd.DataFrame(control_dict, columns=["Gene1", "Gene2"])
        else:
            rows = []

            for complex_name, gene_pairs in control_dict.items():
                for g1, g2 in gene_pairs:
                    rows.append([g1, g2, complex_name])

            genePairDF = pd.DataFrame(rows, columns=["Gene1", "Gene2", "Complex"])

    
    return genePairDF

In [10]:
# initialize a test set
all_complexes = cancer_complexes["Complex Name"].to_list()
test_held_out_complex = all_complexes[26]

positive1 = Ten_positive_controls_1119["positive_control_1"]
positive1_df = convert2genePairDF(positive1, "positive")

negative1 = Fixed_10_negative_controls["negative_controls_1"]
negative1_df = convert2genePairDF(negative1, "negative")

print(f'The identity of the held_out complex for testing is {test_held_out_complex}')

The identity of the held_out complex for testing is ISR/EIF2α–ATF4 complex


In [20]:
matrices = {
    'CRISPR' : CRISPR,
    'Gene_expression' : Gene_expression,
    'shRNA' : shRNA,
    "copy_number": copy_number,
    "Gene_mutation": Gene_mutation

}

matrixNames = list(matrices.keys())
matrix_pairs = list(product(matrixNames, repeat = 2))
print(matrix_pairs)

[('CRISPR', 'CRISPR'), ('CRISPR', 'Gene_expression'), ('CRISPR', 'shRNA'), ('CRISPR', 'copy_number'), ('CRISPR', 'Gene_mutation'), ('Gene_expression', 'CRISPR'), ('Gene_expression', 'Gene_expression'), ('Gene_expression', 'shRNA'), ('Gene_expression', 'copy_number'), ('Gene_expression', 'Gene_mutation'), ('shRNA', 'CRISPR'), ('shRNA', 'Gene_expression'), ('shRNA', 'shRNA'), ('shRNA', 'copy_number'), ('shRNA', 'Gene_mutation'), ('copy_number', 'CRISPR'), ('copy_number', 'Gene_expression'), ('copy_number', 'shRNA'), ('copy_number', 'copy_number'), ('copy_number', 'Gene_mutation'), ('Gene_mutation', 'CRISPR'), ('Gene_mutation', 'Gene_expression'), ('Gene_mutation', 'shRNA'), ('Gene_mutation', 'copy_number'), ('Gene_mutation', 'Gene_mutation')]


In [11]:
# A computation function
def bicor(x, y, c=9.0):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    # Remove missing values
    mask = (~np.isnan(x)) & (~np.isnan(y))
    x = x[mask]
    y = y[mask]

    if len(x) < 3:
        return np.nan

    # Median and MAD
    x_med = np.median(x)
    y_med = np.median(y)

    x_mad = np.median(np.abs(x - x_med))
    y_mad = np.median(np.abs(y - y_med))
    
    if x_mad == 0 or y_mad == 0: 
        return np.nan

    # Standardized distances
    ux = (x - x_med) / (c * x_mad)
    uy = (y - y_med) / (c * y_mad)

    # Weights
    wx = (1 - ux**2)**2
    wy = (1 - uy**2)**2
    wx[np.abs(ux) >= 1] = 0
    wy[np.abs(uy) >= 1] = 0

    # Weighted values
    xw = (x - x_med) * wx
    yw = (y - y_med) * wy

    numerator = np.sum(xw * yw)
    denominator = np.sqrt(np.sum(xw**2) * np.sum(yw**2))

    return numerator / denominator if denominator != 0 else np.nan

In [12]:
def computeAllFeatures(ctrl_df, matrices, matrix_pairs, method):

    rows = []

    # ---- Precompute alignment ----
    pair_data = {}

    for m1_name, m2_name in matrix_pairs:

        df1 = matrices[m1_name]
        df2 = matrices[m2_name]

        commonSamples = df1.columns.intersection(df2.columns)

        pair_data[(m1_name, m2_name)] = {
            "df1": df1[commonSamples],
            "df2": df2[commonSamples]
        }

    # ---- Main loop ----
    for pair_row in ctrl_df.itertuples(index=False):

        gene1 = pair_row.Gene1
        gene2 = pair_row.Gene2

        row = {"Gene_pair": f"{gene1}_{gene2}"}

        for (m1_name, m2_name), info in pair_data.items():

            df1 = info["df1"]
            df2 = info["df2"]

            feature_name = f"{m1_name}_vs_{m2_name}"

            if gene1 not in df1.index or gene2 not in df2.index:
                row[feature_name] = np.nan
                continue

            x = df1.loc[gene1].values
            y = df2.loc[gene2].values

            mask = (~np.isnan(x)) & (~np.isnan(y))
            x = x[mask]
            y = y[mask]

            if len(x) < 3:
                row[feature_name] = np.nan
                continue

            sx = np.std(x)
            sy = np.std(y)

            value = np.nan

            if method == "pearson":
                if sx == 0 or sy == 0:
                    value = 0
                else:
                    value = pearsonr(x, y)[0]

            elif method == "spearman":
                if sx == 0 or sy == 0:
                    value = 0
                else:
                    value = spearmanr(x, y)[0]

            elif method == "bicor":
                if sx == 0 or sy == 0:
                    value = 0
                else:
                    value = bicor(x, y)

            elif method == "mi":
                if sx == 0 or sy == 0:
                    value = 0
                elif len(x) >= 20:
                    value = mutual_info_regression(
                        x.reshape(-1,1), y, random_state=0
                    )[0]
                else:
                    value = 0

            else:
                raise ValueError("Invalid method.")

            if np.isnan(value):
                value = 0

            row[feature_name] = value

        rows.append(row)

    return pd.DataFrame(rows)

In [13]:
# Training and testing set splitting
def trainingAtesting_splitting(held_out_complex, single_pos_ctrl):
    pos_df = convert2genePairDF(single_pos_ctrl)

    train_pos = pos_df[pos_df["Complex"] != held_out_complex]
    test_pos = pos_df[pos_df["Complex"] == held_out_complex]

    train_pos_strings = (train_pos["Gene1"] + "_" + train_pos["Gene2"]).tolist()
    test_pos_strings = (test_pos["Gene1"] + "_" + test_pos["Gene2"]).tolist()

    return train_pos_strings, test_pos_strings

# Choose the negative control for training and testing 
def chooseNegativesAccordingly(single_negative_control, n_train_pos, n_test_pos):
    neg_copy = single_negative_control.copy()   # avoid modifying original list 

    train_neg = neg_copy[0:n_train_pos]
    test_neg = neg_copy[n_train_pos:n_train_pos + n_test_pos]

    train_neg_strings = ["_".join(pair) for pair in train_neg]
    test_neg_strings = ["_".join(pair) for pair in test_neg]


    return train_neg_strings, test_neg_strings

In [14]:
# Do feature selection with only the training set (positive + negative)
def featureSignificance(train_pos_quanDF, train_neg_quanDF):

    feature_names = train_pos_quanDF.columns[1:].to_list()
    feature_significance_summary = {}
    for feature in feature_names:
        pos_vals = train_pos_quanDF[feature]
        neg_vals = train_neg_quanDF[feature]

        pos_vals = np.array(pos_vals)
        neg_vals = np.array(neg_vals)

        pos_vals = pos_vals[~np.isnan(pos_vals)]
        neg_vals = neg_vals[~np.isnan(neg_vals)]

        if len(pos_vals) > 2 and len(neg_vals) > 2:
            stat, p = mannwhitneyu(pos_vals, neg_vals, alternative="two-sided", method="asymptotic")
        else:
            stat, p = np.nan, np.nan

        feature_significance_summary[feature] = (stat, p)

    summary_df = pd.DataFrame(feature_significance_summary)

    return summary_df

In [15]:
# Modify the feature selection function for bicor only
def featureSelectionAPreparation(held_out_complex, single_pos_ctrl, single_neg_ctrl, r_type, matrices, matrix_pairs):
    # Convert the raw format into gene pair data frames
    pos_genePair_df = convert2genePairDF(single_pos_ctrl, "positive")
    neg_genePair_df = convert2genePairDF(single_neg_ctrl, "negative")
    # First compute features 
    pos_quan_df = computeAllFeatures(pos_genePair_df, matrices, matrix_pairs, r_type)
    neg_quan_df = computeAllFeatures(neg_genePair_df, matrices, matrix_pairs, r_type)

    # Split the training and testing
    train_pos_genePairs, test_pos_genePairs = trainingAtesting_splitting(held_out_complex, single_pos_ctrl)
    train_neg_genePairs, test_neg_genePairs = chooseNegativesAccordingly(single_neg_ctrl, len(train_pos_genePairs), len(test_pos_genePairs))

    train_pos_quan = pos_quan_df[pos_quan_df["Gene_pair"].isin(train_pos_genePairs)]
    test_pos_quan = pos_quan_df[pos_quan_df["Gene_pair"].isin(test_pos_genePairs)]

    train_neg_quan = neg_quan_df[neg_quan_df["Gene_pair"].isin(train_neg_genePairs)]
    test_neg_quan = neg_quan_df[neg_quan_df["Gene_pair"].isin(test_neg_genePairs)]

    train_pos_quan = train_pos_quan.dropna()
    train_neg_quan = train_neg_quan.dropna()

    test_pos_quan = test_pos_quan.dropna()
    test_neg_quan = test_neg_quan.dropna()


    # Do the feature selection
    feature_signi_df = featureSignificance(train_pos_quan, train_neg_quan)

    return train_pos_quan, train_neg_quan, test_pos_quan, test_neg_quan, feature_signi_df

In [16]:
# Clean the feature significance data frame
def clean_feature_df(feature_df):
    feature_df = feature_df.T
    feature_df = feature_df.dropna()
    feature_df.columns = ["U-stats", "p-val"]
    # Do a multi-testing adjustment
    feature_df["adj_p-val"] = multipletests(feature_df["p-val"], method="fdr_bh")[1]
    # Retain only the significant features
    significant_features = feature_df[feature_df["adj_p-val"] < 0.05].index.to_list()

    if len(significant_features) == 0:
        significant_features = feature_df.sort_values("adj_p-val", ascending=True).head(3).index.to_list()

    return feature_df, significant_features

In [23]:
# Preparation for model fitting
def ready2model(pc_train_df, nc_train_df, pc_test_df, nc_test_df, features2use):

    pc_train_df = pc_train_df.copy()
    nc_train_df = nc_train_df.copy()
    pc_test_df = pc_test_df.copy()
    nc_test_df = nc_test_df.copy()

    # For all training and testing sets, only keep the significant features
    pc_train_df = pc_train_df[features2use]
    nc_train_df = nc_train_df[features2use]
    pc_test_df = pc_test_df[features2use]
    nc_test_df = nc_test_df[features2use]

    # label the positive and negative sets
    pc_train_df["Y"] = 1
    nc_train_df["Y"] = 0
    pc_test_df["Y"] = 1
    nc_test_df["Y"] = 0

    training_set = pd.concat([pc_train_df, nc_train_df], ignore_index=True)
    testing_set = pd.concat([pc_test_df, nc_test_df], ignore_index=True)

    training_set = training_set.dropna()
    testing_set = testing_set.dropna()

    X_train = training_set.drop(columns=["Y"])
    y_train = training_set["Y"]

    X_test = testing_set.drop(columns=["Y"])
    y_test = testing_set["Y"]


    return X_train, y_train, X_test, y_test

In [18]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, accuracy_score, 
                             roc_auc_score, RocCurveDisplay, classification_report)
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score
from sklearn.metrics import precision_recall_curve, average_precision_score

In [19]:
# A function to save the PR and AUC curves for later benchmarking
def modelFitting(X_train, y_train, X_test, y_test,
                 compute_roc_curve=False,
                 compute_pr_curve=False):

    models = {
        'Gradient Boosting Classifier': GradientBoostingClassifier(),
        'Gaussian Naive Bayes': GaussianNB(),
        'XGBoosting': XGBClassifier(eval_metric='logloss', use_label_encoder=False),
        'Logistic Regression': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression())
        ]),
        'Support Vector Classifier': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', SVC(probability=True, random_state=0))
        ]),
        'MLP Classifier': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', MLPClassifier(max_iter=1000))
        ])
    }

    results = {}

    for name, model in models.items():

        # ---- Fit model ----
        model.fit(X_train, y_train)

        # ---- Predictions ----
        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)

        # ---- Prediction scores ----
        if hasattr(model, "predict_proba"):
            y_score = model.predict_proba(X_test)[:, 1]
        else:
            y_score = model.decision_function(X_test)

        # ---- Metrics ----
        train_acc = accuracy_score(y_train, y_pred_train)
        test_acc = accuracy_score(y_test, y_pred_test)

        precision = precision_score(y_test, y_pred_test, zero_division=0)
        recall = recall_score(y_test, y_pred_test, zero_division=0)
        f1 = f1_score(y_test, y_pred_test, zero_division=0)

        if len(np.unique(y_test)) > 1:
            auc = roc_auc_score(y_test, y_score)
        else:
            auc = np.nan

        model_result = {
            "train_accuracy": train_acc,
            "test_accuracy": test_acc,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "roc_auc": auc,

            # IMPORTANT: store raw outputs for later curves
            "y_test": y_test,
            "y_score": y_score
        }

        # ---- Optional ROC curve ----
        if compute_roc_curve and len(np.unique(y_test)) > 1:
            fpr, tpr, _ = roc_curve(y_test, y_score)
            model_result["fpr"] = fpr
            model_result["tpr"] = tpr

        # ---- Optional PR curve ----
        if compute_pr_curve:
            precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_score)
            model_result["precision_curve"] = precision_curve
            model_result["recall_curve"] = recall_curve

        results[name] = model_result

        print(f"{name} Finished.")

    return results

### Pearson with feature selection

In [24]:
# For each complex, iterate through ten independent positive and negative sets

ten_model_performance = {}
for i in range(10):
    print(f'Now, we are processing trial {i + 1}')
    pc_key = f'positive_control_{i+1}'
    nc_key = f'negative_controls_{i+1}'
    # Get the positive and negative sets
    this_pc = Ten_positive_controls_1119[pc_key]
    this_nc = Fixed_10_negative_controls[nc_key]


    train_pos_quan, train_neg_quan, test_pos_quan, test_neg_quan, feature_df = featureSelectionAPreparation(test_held_out_complex, 
                                                                                                        this_pc, 
                                                                                                        this_nc, 
                                                                                                        "pearson", 
                                                                                                        matrices, matrix_pairs)
    
    feature_df_clean, sig_fea = clean_feature_df(feature_df)

    if len(test_pos_quan) == 0:
        ten_model_performance[f'Trial{i}'] = None

    X_train, y_train, X_test, y_test = ready2model(train_pos_quan, train_neg_quan, test_pos_quan, test_neg_quan, sig_fea)

    ten_model_performance[f'Trial{i}'] = modelFitting(X_train, y_train, X_test, y_test, True, True)

Now, we are processing trial 1
Gradient Boosting Classifier Finished.
Gaussian Naive Bayes Finished.
XGBoosting Finished.
Logistic Regression Finished.
Support Vector Classifier Finished.


/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [15:52:15] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP Classifier Finished.
Now, we are processing trial 2
Gradient Boosting Classifier Finished.
Gaussian Naive Bayes Finished.
XGBoosting Finished.
Logistic Regression Finished.
Support Vector Classifier Finished.


/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [15:52:54] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP Classifier Finished.
Now, we are processing trial 3
Gradient Boosting Classifier Finished.
Gaussian Naive Bayes Finished.
XGBoosting Finished.
Logistic Regression Finished.
Support Vector Classifier Finished.


...

/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [15:56:49] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


MLP Classifier Finished.
Now, we are processing trial 10
Gradient Boosting Classifier Finished.
Gaussian Naive Bayes Finished.
XGBoosting Finished.
Logistic Regression Finished.
Support Vector Classifier Finished.


/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [15:57:21] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


MLP Classifier Finished.


/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


In [26]:
ten_model_performance["Trial9"]

{'Gradient Boosting Classifier': {'train_accuracy': 1.0,
  'test_accuracy': 0.75,
  'precision': 1.0,
  'recall': 0.5,
  'f1_score': 0.6666666666666666,
  'roc_auc': 0.5,
  'y_test': 0    1
  1    1
  2    0
  3    0
  Name: Y, dtype: int64,
  'y_score': array([0.00462263, 0.52893569, 0.01643143, 0.0071632 ]),
  'fpr': array([0., 0., 1., 1.]),
  'tpr': array([0. , 0.5, 0.5, 1. ]),
  'precision_curve': array([0.5       , 0.33333333, 0.5       , 1.        , 1.        ]),
  'recall_curve': array([1. , 0.5, 0.5, 0.5, 0. ])},
 'Gaussian Naive Bayes': {'train_accuracy': 0.8156028368794326,
  'test_accuracy': 0.5,
  'precision': 0.0,
  'recall': 0.0,
  'f1_score': 0.0,
  'roc_auc': 0.25,
  'y_test': 0    1
  1    1
  2    0
  3    0
  Name: Y, dtype: int64,
  'y_score': array([0.08821365, 0.01856307, 0.05965177, 0.09373557]),
  'fpr': array([0. , 0.5, 0.5, 1. , 1. ]),
  'tpr': array([0. , 0. , 0.5, 0.5, 1. ]),
  'precision_curve': array([0.5       , 0.33333333, 0.5       , 0.        , 1.     

In [27]:
test_held_out_complex

'ISR/EIF2α–ATF4 complex'

In [29]:
with open('ISR_EIF2a_ATF4_complex_10pearsonWithSel_performance.pkl', "wb") as f:
    pickle.dump(ten_model_performance, f)

In [30]:
os.chdir('/Users/guowanyi/Desktop/spring 2026/Louis_Vuitton/March 11 update/PearsonWithSel')
os.getcwd()

'/Users/guowanyi/Desktop/spring 2026/Louis_Vuitton/March 11 update/PearsonWithSel'

In [31]:
# Finger Crossed! It will take some time

for complex_name in all_complexes[0:26]:
    print(f'It is the turn for {complex_name}')

    ten_model_performance = {}
    for i in range(10):
        print(f'Now, we are processing trial {i + 1}')
        pc_key = f'positive_control_{i+1}'
        nc_key = f'negative_controls_{i+1}'
        # Get the positive and negative sets
        this_pc = Ten_positive_controls_1119[pc_key]
        this_nc = Fixed_10_negative_controls[nc_key]


        train_pos_quan, train_neg_quan, test_pos_quan, test_neg_quan, feature_df = featureSelectionAPreparation(complex_name, 
                                                                                                            this_pc, 
                                                                                                            this_nc, 
                                                                                                            "pearson", 
                                                                                                            matrices, matrix_pairs)
        
        feature_df_clean, sig_fea = clean_feature_df(feature_df)

        if len(test_pos_quan) == 0:
            ten_model_performance[f'Trial{i}'] = None

        X_train, y_train, X_test, y_test = ready2model(train_pos_quan, train_neg_quan, test_pos_quan, test_neg_quan, sig_fea)

        ten_model_performance[f'Trial{i}'] = modelFitting(X_train, y_train, X_test, y_test, True, True)

    safe_name = complex_name.replace(" ", "_")
    safe_name = re.sub(r'[^A-Za-z0-9_\-]', "", safe_name)

    with open(f'{safe_name}_10pearsonWithSel_performance.pkl', "wb") as f:
        pickle.dump(ten_model_performance, f)

    
    print(f"Saved results for {complex_name}")

It is the turn for Cyclin–CDK complexes
Now, we are processing trial 1
Gradient Boosting Classifier Finished.
Gaussian Naive Bayes Finished.
XGBoosting Finished.
Logistic Regression Finished.
Support Vector Classifier Finished.


/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [16:10:43] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP Classifier Finished.
Now, we are processing trial 2
Gradient Boosting Classifier Finished.
Gaussian Naive Bayes Finished.
XGBoosting Finished.
Logistic Regression Finished.
Support Vector Classifier Finished.


/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [16:11:51] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP Classifier Finished.
Now, we are processing trial 3
Gradient Boosting Classifier Finished.
Gaussian Naive Bayes Finished.
XGBoosting Finished.
Logistic Regression Finished.
Support Vector Classifier Finished.


...

/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [19:43:30] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP Classifier Finished.
Now, we are processing trial 10
Gradient Boosting Classifier Finished.
Gaussian Naive Bayes Finished.


/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [19:44:07] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoosting Finished.
Logistic Regression Finished.
Support Vector Classifier Finished.
MLP Classifier Finished.
Saved results for HSP90–CDC37 chaperone complex


/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


In [32]:
1 + 1

2

### Spearman with feature selection

In [33]:
os.chdir('/Users/guowanyi/Desktop/spring 2026/Louis_Vuitton/March 11 update/SpearmanWithSel')
os.getcwd()

'/Users/guowanyi/Desktop/spring 2026/Louis_Vuitton/March 11 update/SpearmanWithSel'

In [34]:
# Finger Crossed! It will take some time

for complex_name in all_complexes:
    print(f'It is the turn for {complex_name}')

    ten_model_performance = {}
    for i in range(10):
        print(f'Now, we are processing trial {i + 1}')
        pc_key = f'positive_control_{i+1}'
        nc_key = f'negative_controls_{i+1}'
        # Get the positive and negative sets
        this_pc = Ten_positive_controls_1119[pc_key]
        this_nc = Fixed_10_negative_controls[nc_key]


        train_pos_quan, train_neg_quan, test_pos_quan, test_neg_quan, feature_df = featureSelectionAPreparation(complex_name, 
                                                                                                            this_pc, 
                                                                                                            this_nc, 
                                                                                                            "spearman", 
                                                                                                            matrices, matrix_pairs)
        
        feature_df_clean, sig_fea = clean_feature_df(feature_df)

        if len(test_pos_quan) == 0:
            ten_model_performance[f'Trial{i}'] = None

        X_train, y_train, X_test, y_test = ready2model(train_pos_quan, train_neg_quan, test_pos_quan, test_neg_quan, sig_fea)

        ten_model_performance[f'Trial{i}'] = modelFitting(X_train, y_train, X_test, y_test, True, True)

    safe_name = complex_name.replace(" ", "_")
    safe_name = re.sub(r'[^A-Za-z0-9_\-]', "", safe_name)

    with open(f'{safe_name}_10spearmanWithSel_performance.pkl', "wb") as f:
        pickle.dump(ten_model_performance, f)

    
    print(f"Saved results for {complex_name}")

It is the turn for Cyclin–CDK complexes
Now, we are processing trial 1
Gradient Boosting Classifier Finished.
Gaussian Naive Bayes Finished.
XGBoosting Finished.
Logistic Regression Finished.
Support Vector Classifier Finished.


/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [10:04:53] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP Classifier Finished.
Now, we are processing trial 2
Gradient Boosting Classifier Finished.
Gaussian Naive Bayes Finished.
XGBoosting Finished.
Logistic Regression Finished.
Support Vector Classifier Finished.


/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [10:05:24] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP Classifier Finished.
Now, we are processing trial 3
Gradient Boosting Classifier Finished.
Gaussian Naive Bayes Finished.
XGBoosting Finished.
Logistic Regression Finished.
Support Vector Classifier Finished.


...

/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [12:28:54] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP Classifier Finished.
Now, we are processing trial 10
Gradient Boosting Classifier Finished.
Gaussian Naive Bayes Finished.
XGBoosting Finished.
Logistic Regression Finished.
Support Vector Classifier Finished.


/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [12:29:27] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


MLP Classifier Finished.
Saved results for ISR/EIF2α–ATF4 complex


/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


### Mutual Information with feature selection

In [35]:
1 + 1

2

In [36]:
os.chdir('/Users/guowanyi/Desktop/spring 2026/Louis_Vuitton/March 11 update/MiWithSel')
os.getcwd()

'/Users/guowanyi/Desktop/spring 2026/Louis_Vuitton/March 11 update/MiWithSel'

In [37]:
# Finger Crossed! It will take some time

for complex_name in all_complexes:
    print(f'It is the turn for {complex_name}')

    ten_model_performance = {}
    for i in range(10):
        print(f'Now, we are processing trial {i + 1}')
        pc_key = f'positive_control_{i+1}'
        nc_key = f'negative_controls_{i+1}'
        # Get the positive and negative sets
        this_pc = Ten_positive_controls_1119[pc_key]
        this_nc = Fixed_10_negative_controls[nc_key]


        train_pos_quan, train_neg_quan, test_pos_quan, test_neg_quan, feature_df = featureSelectionAPreparation(complex_name, 
                                                                                                            this_pc, 
                                                                                                            this_nc, 
                                                                                                            "mi", 
                                                                                                            matrices, matrix_pairs)
        
        feature_df_clean, sig_fea = clean_feature_df(feature_df)

        if len(test_pos_quan) == 0:
            ten_model_performance[f'Trial{i}'] = None

        X_train, y_train, X_test, y_test = ready2model(train_pos_quan, train_neg_quan, test_pos_quan, test_neg_quan, sig_fea)

        ten_model_performance[f'Trial{i}'] = modelFitting(X_train, y_train, X_test, y_test, True, True)

    safe_name = complex_name.replace(" ", "_")
    safe_name = re.sub(r'[^A-Za-z0-9_\-]', "", safe_name)

    with open(f'{safe_name}_10miWithSel_performance.pkl', "wb") as f:
        pickle.dump(ten_model_performance, f)

    
    print(f"Saved results for {complex_name}")

It is the turn for Cyclin–CDK complexes
Now, we are processing trial 1
Gradient Boosting Classifier Finished.
Gaussian Naive Bayes Finished.
XGBoosting Finished.
Logistic Regression Finished.
Support Vector Classifier Finished.


/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [15:02:31] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


MLP Classifier Finished.
Now, we are processing trial 2
Gradient Boosting Classifier Finished.
Gaussian Naive Bayes Finished.
XGBoosting Finished.
Logistic Regression Finished.
Support Vector Classifier Finished.
MLP Classifier Finished.
Now, we are processing trial 3


/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [15:03:11] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Gradient Boosting Classifier Finished.
Gaussian Naive Bayes Finished.
XGBoosting Finished.
Logistic Regression Finished.
Support Vector Classifier Finished.
MLP Classifier Finished.
Now, we are processing trial 4


...

Gradient Boosting Classifier Finished.
Gaussian Naive Bayes Finished.
XGBoosting Finished.
Logistic Regression Finished.
Support Vector Classifier Finished.
MLP Classifier Finished.
Now, we are processing trial 10


/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [19:23:19] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Gradient Boosting Classifier Finished.
Gaussian Naive Bayes Finished.
XGBoosting Finished.
Logistic Regression Finished.
Support Vector Classifier Finished.


/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [19:24:04] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


MLP Classifier Finished.
Saved results for ISR/EIF2α–ATF4 complex


In [38]:
1 + 1

2

### Bicor with feature selection

In [39]:
bicor_features_pairs = [
    ("CRISPR", "CRISPR"),
    ('CRISPR', 'Gene_expression'),
    ('CRISPR', 'shRNA'),
    ('Gene_expression', 'CRISPR'),
    ('Gene_expression', 'Gene_expression'),
    ('Gene_expression', 'shRNA'),
    ('shRNA', 'CRISPR'),
    ('shRNA', 'Gene_expression'),
    ('shRNA', 'shRNA')
]

In [40]:
os.chdir('/Users/guowanyi/Desktop/spring 2026/Louis_Vuitton/March 11 update/BicorWithSel')
os.getcwd()

'/Users/guowanyi/Desktop/spring 2026/Louis_Vuitton/March 11 update/BicorWithSel'

In [42]:
# Finger Crossed! It will take some time

for complex_name in all_complexes:
    print(f'It is the turn for {complex_name}')

    ten_model_performance = {}
    for i in range(10):
        print(f'Now, we are processing trial {i + 1}')
        pc_key = f'positive_control_{i+1}'
        nc_key = f'negative_controls_{i+1}'
        # Get the positive and negative sets
        this_pc = Ten_positive_controls_1119[pc_key]
        this_nc = Fixed_10_negative_controls[nc_key]


        train_pos_quan, train_neg_quan, test_pos_quan, test_neg_quan, feature_df = featureSelectionAPreparation(complex_name, 
                                                                                                            this_pc, 
                                                                                                            this_nc, 
                                                                                                            "bicor", 
                                                                                                            matrices, bicor_features_pairs)
        
        feature_df_clean, sig_fea = clean_feature_df(feature_df)

        X_train, y_train, X_test, y_test = ready2model(train_pos_quan, train_neg_quan, test_pos_quan, test_neg_quan, sig_fea)

        if len(X_test) == 0:
            ten_model_performance[f'Trial{i}'] = None
            continue

        ten_model_performance[f'Trial{i}'] = modelFitting(X_train, y_train, X_test, y_test, True, True)

    safe_name = complex_name.replace(" ", "_")
    safe_name = re.sub(r'[^A-Za-z0-9_\-]', "", safe_name)

    with open(f'{safe_name}_10bicorWithSel_performance.pkl', "wb") as f:
        pickle.dump(ten_model_performance, f)

    
    print(f"Saved results for {complex_name}")

It is the turn for Cyclin–CDK complexes
Now, we are processing trial 1
Gradient Boosting Classifier Finished.
Gaussian Naive Bayes Finished.
XGBoosting Finished.
Logistic Regression Finished.
Support Vector Classifier Finished.


/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [09:44:33] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


MLP Classifier Finished.
Now, we are processing trial 2
Gradient Boosting Classifier Finished.
Gaussian Naive Bayes Finished.
XGBoosting Finished.
Logistic Regression Finished.
Support Vector Classifier Finished.


/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [09:44:35] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP Classifier Finished.
Now, we are processing trial 3
Gradient Boosting Classifier Finished.
Gaussian Naive Bayes Finished.
XGBoosting Finished.
Logistic Regression Finished.
Support Vector Classifier Finished.


...

/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [09:49:49] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


MLP Classifier Finished.
Now, we are processing trial 10
Gradient Boosting Classifier Finished.
Gaussian Naive Bayes Finished.
XGBoosting Finished.
Logistic Regression Finished.
Support Vector Classifier Finished.


/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [09:49:50] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


MLP Classifier Finished.
Saved results for ISR/EIF2α–ATF4 complex


/Users/guowanyi/miniconda3/envs/ML/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
